In [1]:
import os, sys
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, Markdown, Math
import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import norm
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Versión local de la librería: se busca src/pmule.py subiendo desde el directorio actual
_aqui = os.path.abspath('.')
while _aqui != os.path.dirname(_aqui) and not os.path.isfile(os.path.join(_aqui, 'src', 'pmule.py')):
    _aqui = os.path.dirname(_aqui)
sys.path.append(os.path.join(_aqui, 'src'))
from pmule import (PredecessorTable, ProjectGraph, EarnedValue,
                   pinta_tikz, plot_cargas, plot_curvas_valor_ganado,
                   tanto_por_uno, ordena_rutas)

GANTT_PARAMS = dict(arrows=True)   # flechas de prelación en los diagramas de Gantt (rojas las críticas)

In [2]:
# Funciones auxiliares para mostrar los cálculos intermedios paso a paso
from itertools import combinations as _combinations


def detalle_tiempos_nodos(proyecto, duraciones):
    """Tiempos tempranos y tardíos de los nodos con los candidatos que se comparan en cada uno."""
    G = proyecto.pert_graph
    d = {data['activity']: (0 if data['activity'].startswith('@') else int(duraciones[data['activity']]))
         for _, _, data in G.edges(data=True)}
    nodos = list(proyecto.nodes)
    early, filas = {}, []
    for j in nodos:
        entrantes = [(i, data['activity']) for i, _, data in G.in_edges(j, data=True)]
        if not entrantes:
            early[j] = 0
            filas.append({'nodo': j, 'candidatos t_i + d_ij': 'nodo inicial', 'tiempo temprano = máx': 0})
            continue
        cand = {f'{a}: {early[i]} + {d[a]} = {early[i] + d[a]}': early[i] + d[a] for i, a in entrantes}
        early[j] = max(cand.values())
        filas.append({'nodo': j, 'candidatos t_i + d_ij': '; '.join(cand), 'tiempo temprano = máx': early[j]})
    tempranos = pd.DataFrame(filas).set_index('nodo')
    late, filas = {}, []
    for i in reversed(nodos):
        salientes = [(j, data['activity']) for _, j, data in G.out_edges(i, data=True)]
        if not salientes:
            late[i] = early[i]
            filas.append({'nodo': i, 'candidatos t*_j − d_ij': 'nodo final (= su tiempo temprano)', 'tiempo tardío = mín': late[i]})
            continue
        cand = {f'{a}: {late[j]} − {d[a]} = {late[j] - d[a]}': late[j] - d[a] for j, a in salientes}
        late[i] = min(cand.values())
        filas.append({'nodo': i, 'candidatos t*_j − d_ij': '; '.join(cand), 'tiempo tardío = mín': late[i]})
    tardios = pd.DataFrame(filas).set_index('nodo')
    return tempranos, tardios


def detalle_holguras(calendario, prelaciones):
    """Cómo se obtiene cada fecha y cada holgura a partir de las vecinas."""
    pred = prelaciones.immediate_predecessor
    suc = {a: sorted(s for s, ps in pred.items() if a in ps) for a in calendario.index}
    fin = int(calendario['fin_mas_temprano'].max())
    filas = []
    for a in calendario.index:
        ES, EF = int(calendario.loc[a, 'inicio_mas_temprano']), int(calendario.loc[a, 'fin_mas_temprano'])
        LS, LF = int(calendario.loc[a, 'inicio_mas_tardio']), int(calendario.loc[a, 'fin_mas_tardio'])
        dur = int(calendario.loc[a, 'duracion'])
        ps, ss = sorted(pred[a]), suc[a]
        ef_pred = {p: int(calendario.loc[p, 'fin_mas_temprano']) for p in ps}
        es_suc = {s: int(calendario.loc[s, 'inicio_mas_temprano']) for s in ss}
        lf_pred = {p: int(calendario.loc[p, 'fin_mas_tardio']) for p in ps}
        ls_suc = {s: int(calendario.loc[s, 'inicio_mas_tardio']) for s in ss}
        min_es_suc = min(es_suc.values()) if es_suc else fin
        max_lf_pred = max(lf_pred.values()) if lf_pred else 0
        filas.append({
            'actividad': a,
            'ES = máx EF pred.': (f"máx({', '.join(f'{p}: {v}' for p, v in ef_pred.items())}) = {ES}" if ps else f'inicial → {ES}'),
            'EF = ES + d': f'{ES} + {dur} = {EF}',
            'LF = mín LS suc.': (f"mín({', '.join(f'{s}: {v}' for s, v in ls_suc.items())}) = {LF}" if ss else f'final → {LF}'),
            'LS = LF − d': f'{LF} − {dur} = {LS}',
            'H_total = LS − ES': f'{LS} − {ES} = {LS - ES}',
            'H_libre = mín ES suc. − EF': f"{min_es_suc}{'' if ss else ' (fin)'} − {EF} = {min_es_suc - EF}",
            'H_indep. = máx(0, mín ES suc. − máx LF pred. − d)': f'máx(0, {min_es_suc} − {max_lf_pred} − {dur}) = {max(0, min_es_suc - max_lf_pred - dur)}',
        })
    return pd.DataFrame(filas).set_index('actividad')


def detalle_rutas(R, valores, nombre='duración', decimales=0):
    """Suma término a término de un valor (duraciones, varianzas…) a lo largo de cada ruta."""
    filas = {}
    for r in R.index:
        acts = list(R.columns[R.loc[r] == 1])
        total = float(sum(valores[a] for a in acts))
        filas[r] = {'actividades': ' - '.join(acts),
                    f'{nombre} = suma': ' + '.join(f'{a}({valores[a]:.{decimales}f})' for a in acts) + f' = {total:.{decimales}f}',
                    nombre: total}
    return pd.DataFrame(filas).T


def detalle_estimaciones(datos, a='a', m='m', b='b'):
    """Media y varianza PERT de cada actividad con los números sustituidos."""
    filas = {}
    for act, fila in datos.iterrows():
        A, M, B = float(fila[a]), float(fila[m]), float(fila[b])
        filas[act] = {'D = (a + 4m + b)/6': f'({A:g} + 4·{M:g} + {B:g})/6 = {(A + 4 * M + B) / 6:.2f}',
                      'σ² = ((b − a)/6)²': f'(({B:g} − {A:g})/6)² = {((B - A) / 6) ** 2:.3f}'}
    return pd.DataFrame(filas).T


def detalle_ackoff(proyecto, duraciones, topes, cur, pasos):
    """En cada paso de Ackoff-Sasieni: rutas críticas, opciones consideradas (actividades comunes y parejas)
    con su coste, y la elegida."""
    R = proyecto.path_matrix(dummies=False)
    d = duraciones.astype(float).copy()
    filas = []
    for paso, elegidas in sorted(pasos.items()):
        L = R @ d[R.columns]
        T = float(L.max())
        criticas = [r for r in L.index if np.isclose(L[r], T)]
        conjuntos = [set(R.columns[R.loc[r] == 1]) for r in criticas]
        con_margen = {x for x in R.columns if d[x] > topes[x] and np.isfinite(cur[x])}
        opciones = {x: float(cur[x]) for x in set.intersection(*conjuntos) & con_margen}
        if len(conjuntos) > 1:
            cand = sorted(set().union(*conjuntos) & con_margen)
            for p, q in _combinations(cand, 2):
                if all(p in s or q in s for s in conjuntos) and p not in opciones and q not in opciones:
                    opciones[f'{p}+{q}'] = float(cur[p] + cur[q])
        orden = sorted(opciones.items(), key=lambda kv: kv[1])
        filas.append({'paso': paso + 1, 'duración': int(T), 'rutas críticas': ', '.join(criticas),
                      'opciones (coste)': '; '.join(f'{k} ({v:.0f})' for k, v in orden[:6]) + (' …' if len(orden) > 6 else ''),
                      'elegida': ', '.join(elegidas), 'coste del paso': float(sum(cur[x] for x in elegidas))})
        for x in elegidas:
            d[x] -= 1
    return pd.DataFrame(filas).set_index('paso')


def detalle_valor_ganado(datos_vg, calendario, semana):
    """PV, EV y AC actividad a actividad en el punto de control."""
    filas = {}
    for a, fila in datos_vg.iterrows():
        es, dur = int(calendario.loc[a, 'inicio_mas_temprano']), int(calendario.loc[a, 'duracion'])
        frac = min(max(semana - es, 0), dur) / dur
        filas[a] = {'BAC': float(fila['BAC']), 'planificado hasta la semana': f'{min(max(semana - es, 0), dur)} de {dur}',
                    'PV = BAC·fracción': f'{fila["BAC"]:.0f}·{frac:.2f} = {fila["BAC"] * frac:.1f}',
                    'realizado (%)': int(fila['performed']),
                    'EV = BAC·%': f'{fila["BAC"]:.0f}·{fila["performed"] / 100:.2f} = {fila["BAC"] * fila["performed"] / 100:.1f}',
                    'AC': float(fila['AC']), 'PV_i': fila['BAC'] * frac, 'EV_i': fila['BAC'] * fila['performed'] / 100}
    t = pd.DataFrame(filas).T
    return t


def muestra_formula(nombre, expresion, valor, unidades=''):
    """Una línea «nombre = expresión = valor» en Markdown."""
    return Markdown(f'**{nombre}** = {expresion} = **{valor}**{(" " + unidades) if unidades else ""}')

In [3]:
%%capture
%mkdir -p ~/temp
%cd ~/temp

# Enunciado

Considere el proyecto cuyo cuadro de prelaciones, duraciones (semanas) y operarios necesarios se adjunta, con 4 operarios disponibles. Se discute si merece la pena organizar el trabajo de modo que una actividad pueda interrumpirse y reanudarse más tarde (sin coste añadido) cuando los operarios hagan falta en otra más urgente.

In [4]:
pregunta_1 = "1. Programe el proyecto con un planificador en paralelo y la regla «menor holgura total primero» suponiendo que las actividades **no** pueden interrumpirse. ¿Cuánto dura?  "
pregunta_2 = "2. Repita suponiendo que sí pueden interrumpirse: en cada semana se vuelven a ordenar *todas* las actividades disponibles, también las empezadas, y una actividad en curso puede quedar parada si entra otra más prioritaria. ¿Cuánto dura? ¿Qué actividades se interrumpen y cuántas veces?  "
pregunta_3 = "3. Compare las dos programaciones semana a semana. ¿De dónde sale la ganancia (si la hay)?  "
pregunta_4 = "4. ¿Qué costes ocultos tiene interrumpir una actividad en la práctica? ¿Cuándo compensa?  "

for item in sorted(dir()):
    if item.startswith('pregunta_'):
        display(Markdown(eval(item)))

1. Programe el proyecto con un planificador en paralelo y la regla «menor holgura total primero» suponiendo que las actividades **no** pueden interrumpirse. ¿Cuánto dura?  

2. Repita suponiendo que sí pueden interrumpirse: en cada semana se vuelven a ordenar *todas* las actividades disponibles, también las empezadas, y una actividad en curso puede quedar parada si entra otra más prioritaria. ¿Cuánto dura? ¿Qué actividades se interrumpen y cuántas veces?  

3. Compare las dos programaciones semana a semana. ¿De dónde sale la ganancia (si la hay)?  

4. ¿Qué costes ocultos tiene interrumpir una actividad en la práctica? ¿Cuándo compensa?  

In [5]:
#| tbl-cap: "Cuadro de datos del enunciado"
#| label: tbl-datosEnunciado
datos_enunciado = pd.DataFrame([
#  activity, predecessor, duration, resources
    ('A'     , '---'      , 6       , 4        ),
    ('B'     , '---'      , 5       , 1        ),
    ('C'     , 'A,B'      , 3       , 4        ),
    ('D'     , 'C'        , 6       , 2        ),
    ('E'     , 'B'        , 4       , 1        ),
    ('F'     , 'C'        , 3       , 1        ),
    ('G'     , 'E,F'      , 2       , 4        ),
    ('H'     , 'D,G'      , 3       , 4        ),
], columns=['activity', 'predecessor', 'duration', 'resources']).set_index('activity')
datos_enunciado

,predecessor,duration,resources
activity,,,
A,---,6,4
B,---,5,1
C,"A,B",3,4
D,C,6,2
E,B,4,1
F,C,3,1
G,"E,F",2,4
H,"D,G",3,4


# Resolución

In [6]:
datos_proyecto = datos_enunciado.copy()
prelaciones = PredecessorTable.from_dataframe_of_strings(datos_proyecto, activity='activity', predecessor='predecessor')
proyecto = prelaciones.create_project()
calendario = proyecto.calendar(datos_proyecto['duration'])
duracion_minima = proyecto.duration(datos_proyecto['duration'])
limite = 4

In [7]:
def planificar(datos, prelaciones, limite, prioridad, interrumpible=False, max_periodos=200):
    '''Programación en paralelo periodo a periodo con un recurso renovable limitado.
    prioridad(a, t) -> número: menor = antes. Devuelve (semanas trabajadas por actividad, inicio, fin).'''
    pred = prelaciones.immediate_predecessor
    restante = datos['duration'].astype(int).to_dict()
    rec = datos['resources'].astype(int).to_dict()
    trabajo = {a: [] for a in restante}
    en_curso = set()
    for t in range(1, max_periodos + 1):
        terminadas = {a for a, r in restante.items() if r == 0}
        if len(terminadas) == len(restante):
            break
        elegibles = [a for a in restante if restante[a] > 0 and set(pred[a]) <= terminadas]
        if not interrumpible:
            # las ya empezadas siguen; las nuevas se ordenan por prioridad
            fijas = [a for a in elegibles if a in en_curso]
            nuevas = sorted([a for a in elegibles if a not in en_curso], key=lambda a: prioridad(a, t))
            orden = fijas + nuevas
        else:
            orden = sorted(elegibles, key=lambda a: prioridad(a, t))
        disponible = limite
        activas = []
        for a in orden:
            if rec[a] <= disponible:
                activas.append(a); disponible -= rec[a]
        for a in activas:
            restante[a] -= 1; trabajo[a].append(t)
        en_curso = {a for a in activas if restante[a] > 0}
    inicio = pd.Series({a: (min(s) if s else np.nan) for a, s in trabajo.items()})
    fin = pd.Series({a: (max(s) if s else np.nan) for a, s in trabajo.items()})
    return trabajo, inicio, fin

def gantt_texto(trabajo, limite):
    '''Tabla actividad × semana con el recurso usado y fila de carga total.'''
    T = max(max(s) for s in trabajo.values() if s)
    g = pd.DataFrame('', index=list(trabajo), columns=range(1, T + 1))
    for a, semanas in trabajo.items():
        for t in semanas:
            g.loc[a, t] = int(datos_proyecto.loc[a, 'resources'])
    carga = pd.Series({t: sum(int(datos_proyecto.loc[a, 'resources']) for a, s in trabajo.items() if t in s) for t in g.columns})
    g.loc['Total'] = carga
    g.loc['≤ límite'] = ['sí' if c <= limite else 'NO' for c in carga]
    return g

In [8]:
regla = lambda a, t: (calendario.loc[a, 'H_total'], datos_proyecto.loc[a, 'duration'], a)

## Apartado 1

In [9]:
display(Markdown(pregunta_1))

1. Programe el proyecto con un planificador en paralelo y la regla «menor holgura total primero» suponiendo que las actividades **no** pueden interrumpirse. ¿Cuánto dura?  

In [10]:
#| tbl-cap: "Programación sin interrupciones"
#| label: tbl-sinInterrupcion
trabajo_no, inicio_no, fin_no = planificar(datos_proyecto, prelaciones, limite, regla, interrumpible=False)
gantt_texto(trabajo_no, limite)

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25
A,4,4,4,4,4,4,,,,,,,,,,,,,,,,,,,
B,,,,,,,1,1,1,1,1,,,,,,,,,,,,,,
C,,,,,,,,,,,,4,4,4,,,,,,,,,,,
D,,,,,,,,,,,,,,,2,2,2,2,2,2,,,,,
E,,,,,,,,,,,,,,,1,1,1,1,,,,,,,
F,,,,,,,,,,,,,,,1,1,1,,,,,,,,
G,,,,,,,,,,,,,,,,,,,,,4,4,,,
H,,,,,,,,,,,,,,,,,,,,,,,4,4,4
Total,4,4,4,4,4,4,1,1,1,1,1,4,4,4,4,4,4,3,2,2,4,4,4,4,4
≤ límite,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí


In [11]:
Markdown(f'Sin interrupciones el proyecto dura {int(fin_no.max())} semanas (mínimo sin límite de recursos: {duracion_minima}).')

Sin interrupciones el proyecto dura 25 semanas (mínimo sin límite de recursos: 18).

## Apartado 2

In [12]:
display(Markdown(pregunta_2))

2. Repita suponiendo que sí pueden interrumpirse: en cada semana se vuelven a ordenar *todas* las actividades disponibles, también las empezadas, y una actividad en curso puede quedar parada si entra otra más prioritaria. ¿Cuánto dura? ¿Qué actividades se interrumpen y cuántas veces?  

In [13]:
#| tbl-cap: "Programación con interrupciones permitidas"
#| label: tbl-conInterrupcion
trabajo_si, inicio_si, fin_si = planificar(datos_proyecto, prelaciones, limite, regla, interrumpible=True)
gantt_texto(trabajo_si, limite)

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25
A,4,4,4,4,4,4,,,,,,,,,,,,,,,,,,,
B,,,,,,,1,1,1,1,1,,,,,,,,,,,,,,
C,,,,,,,,,,,,4,4,4,,,,,,,,,,,
D,,,,,,,,,,,,,,,2,2,2,2,2,2,,,,,
E,,,,,,,,,,,,,,,1,1,1,1,,,,,,,
F,,,,,,,,,,,,,,,1,1,1,,,,,,,,
G,,,,,,,,,,,,,,,,,,,,,4,4,,,
H,,,,,,,,,,,,,,,,,,,,,,,4,4,4
Total,4,4,4,4,4,4,1,1,1,1,1,4,4,4,4,4,4,3,2,2,4,4,4,4,4
≤ límite,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí,sí


In [14]:
def interrupciones(semanas):
    return sum(1 for s1, s2 in zip(semanas, semanas[1:]) if s2 > s1 + 1)
inter = pd.Series({a: interrupciones(s) for a, s in trabajo_si.items()})
inter = inter[inter > 0]
Markdown(f'''Con interrupciones el proyecto dura {int(fin_si.max())} semanas. {('Se interrumpen ' + ', '.join(f'{a} ({v} vez/veces)' for a, v in inter.items()) + '.') if len(inter) else 'Ninguna actividad llega a interrumpirse.'}''')

Con interrupciones el proyecto dura 25 semanas. Ninguna actividad llega a interrumpirse.

## Apartado 3

In [15]:
display(Markdown(pregunta_3))

3. Compare las dos programaciones semana a semana. ¿De dónde sale la ganancia (si la hay)?  

In [16]:
#| tbl-cap: "Inicio y fin de cada actividad en las dos programaciones"
#| label: tbl-comparacion
pd.DataFrame({'inicio (no interr.)': inicio_no.astype(int), 'fin (no interr.)': fin_no.astype(int),
              'inicio (interr.)': inicio_si.astype(int), 'fin (interr.)': fin_si.astype(int),
              'semanas trabajadas (interr.)': {a: len(s) for a, s in trabajo_si.items()}})

,inicio (no interr.),fin (no interr.),inicio (interr.),fin (interr.),semanas trabajadas (interr.)
A,1,6,1,6,6
B,7,11,7,11,5
C,12,14,12,14,3
D,15,20,15,20,6
E,15,18,15,18,4
F,15,17,15,17,3
G,21,22,21,22,2
H,23,25,23,25,3


Si hay ganancia, viene de que al permitir interrumpir no hay que esperar a que termine una actividad poco urgente para liberar operarios: una actividad crítica puede entrar en cuanto está disponible. Si no la hay, es que la regla de prioridad ya había evitado bloquear el recurso con actividades holgadas. Interrumpir nunca empeora la duración con este planificador (el no interrumpible es un caso particular), pero puede fragmentar mucho el trabajo.

## Apartado 4

In [17]:
display(Markdown(pregunta_4))

4. ¿Qué costes ocultos tiene interrumpir una actividad en la práctica? ¿Cuándo compensa?  

Interrumpir tiene costes que el modelo ignora: volver a montar y desmontar (tiempos de preparación), pérdida de ritmo y de aprendizaje, el material en curso inmovilizado, la coordinación de los operarios y la dificultad de seguimiento. Compensa cuando el ahorro de plazo vale más que esos costes: actividades largas, modulares (pueden partirse en lotes) y sin costes de preparación apreciables; no compensa en tareas con fraguados, curados o procesos continuos.